# COS30018 Week 1 Lab
## Introduction to LLM Access: Local and Cloud

**Objective:** understand what an LLM API call is, then get local (Ollama) and cloud (Gemini) access working and compare them.

**Before running:** start Ollama in a terminal with `ollama serve`. Then run all cells top to bottom (Kernel → Restart & Run All).

## 1. Imports

In [1]:
import time
import pandas as pd
import ollama
from google import genai
from google.genai import errors
from getpass import getpass

/Users/zadeedhaque/Desktop/COS30018/Week1/venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/zadeedhaque/Desktop/COS30018/Week1/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/zadeedhaque/Desktop/COS30018/Week1/venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or fea

## 2. Local model (Ollama)
The request goes to a server on my own machine (`localhost:11434`). The model weights are stored on my laptop, so no internet is needed.

In [2]:
r = ollama.chat(model='llama3.2:1b',
                messages=[{'role': 'user', 'content': 'What is an intelligent system?'}])
print(r['message']['content'])

An intelligent system, also known as a machine intelligence or artificial intelligence (AI), is a computer system that can perform tasks that typically require human intelligence, such as:

1. Reasoning and problem-solving
2. Learning from data and experience
3. Perception and understanding of language and visual information
4. Making decisions based on input and output

Intelligent systems can be designed to perform a wide range of tasks, including but not limited to:

* Natural Language Processing (NLP): understanding and generating human-like text
* Image recognition: identifying objects, people, and patterns in images
* Speech recognition: transcribing spoken words into text
* Robotics: controlling physical devices and systems
* Game playing: competing with humans in games such as chess, Go, or shogi

Intelligent systems can be categorized into different types, including:

1. Narrow or weak intelligence: designed to perform a specific task, but lacks general intelligence.
2. Genera

## 3. Cloud model (Gemini)
The key is typed into a hidden box with `getpass`, so it is **never saved** in the notebook or pushed to GitHub.

In [3]:
GEMINI_API_KEY = getpass("Paste Gemini key: ").strip()
client = genai.Client(api_key=GEMINI_API_KEY)

# Safe check: shows only length and first 4 characters
print(len(GEMINI_API_KEY), GEMINI_API_KEY[:4])
if not GEMINI_API_KEY.startswith("AIza"):
    print("Warning: Gemini API keys from AI Studio usually start with 'AIza'.")

Paste Gemini key:  ········


53 AQ.A


In [4]:
r = client.models.generate_content(model='gemini-3.8-flash',
                                   contents='What is an intelligent system?')
print(r.text)

An **intelligent system** is a technologically advanced machine, software program, or network that can perceive its environment, process information, learn from experience, and take autonomous actions to achieve specific goals. 

Unlike traditional computer programs—which simply follow rigid, pre-written instructions—an intelligent system possesses capabilities inspired by human cognition, such as **reasoning, problem-solving, pattern recognition, and adaptability.**

---

### Core Characteristics of an Intelligent System

To be considered "intelligent," a system typically relies on a continuous loop of four key functions:

1. **Perception (Sensing):** 
   The system collects data from its environment using sensors, cameras, microphones, or digital inputs (e.g., text, databases).
2. **Reasoning and Processing (Thinking):**
   It analyzes the collected data using algorithms, logic, or models to understand context, solve problems, and make decisions.
3. **Action (Actuation):**
   It prod

## 4. Helper functions
`ask_cloud` retries if Google's servers are busy (503 error).

In [5]:
def ask_local(prompt):
    response = ollama.chat(
        model='llama3.2:1b',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content']


def ask_cloud(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model='gemini-3.8-flash',
                contents=prompt
            )
            return response.text
        except errors.ServerError:          # 5xx: server busy, wait and retry
            time.sleep(5 * (attempt + 1))
    return "ERROR: Gemini unavailable"


def timed(fn, prompt):
    start = time.time()
    out = fn(prompt)
    return out, time.time() - start

## 5. Load the dataset
`Text` = the review, `Sentiment` = the true label (0 = negative, 1 = positive).

In [6]:
df = pd.read_csv('reviews.csv')
print(df.shape)
print(df['Sentiment'].value_counts())
df.head()

(99, 2)
Sentiment
0    99
Name: count, dtype: int64


,Text,Sentiment
0,Once again Mr. Costner has dragged out a movie...,0
1,This is an example of why the majority of acti...,0
2,"First of all I hate those moronic rappers, who...",0
3,Not even the Beatles could write songs everyon...,0
4,Brass pictures (movies is not a fitting word f...,0


## 6. Classify reviews with both models and time each call
Only 10 reviews, with a pause, to stay under Gemini's free-tier rate limit.

In [7]:
def to_label(answer):
    a = answer.lower()
    if 'positive' in a: return 1
    if 'negative' in a: return 0
    return None

results = []
for text, true in zip(df['Text'].head(10), df['Sentiment'].head(10)):
    prompt = f"Is this review positive or negative? Answer with one word.\n\n{text}"
    local_out, local_t = timed(ask_local, prompt)
    cloud_out, cloud_t = timed(ask_cloud, prompt)
    results.append({'review': text[:60], 'true': true,
                    'local': local_out.strip(), 'local_s': round(local_t, 2),
                    'cloud': cloud_out.strip(), 'cloud_s': round(cloud_t, 2)})
    time.sleep(4)

res = pd.DataFrame(results)
res['local_label'] = res['local'].apply(to_label)
res['cloud_label'] = res['cloud'].apply(to_label)
res

,review,true,local,local_s,cloud,cloud_s,local_label,cloud_label
0,Once again Mr. Costner has dragged out a movie...,0,Negative,0.43,Negative,2.67,0,0
1,This is an example of why the majority of acti...,0,Negative.,0.55,Negative,2.99,0,0
2,"First of all I hate those moronic rappers, who...",0,Negative,0.49,Negative,40.51,0,0
3,Not even the Beatles could write songs everyon...,0,Negative,0.59,Negative,14.43,0,0
4,Brass pictures (movies is not a fitting word f...,0,Negative,0.55,Negative,20.02,0,0
5,"A funny thing happened to me while watching ""M...",0,Negative.,0.46,Negative,33.68,0,0
6,This German horror film has to be one of the w...,0,Negative,0.46,Negative,58.41,0,0
7,"Being a long-time fan of Japanese film, I expe...",0,Negative.,0.44,Negative,46.44,0,0
8,"""Tokyo Eyes"" tells of a 17 year old Japanese g...",0,Negative.,0.42,Negative,61.62,0,0
9,Wealthy horse ranchers in Buenos Aires have a ...,0,Negative.,0.48,Negative,40.91,0,0


## 7. Summary numbers for the comparison table

In [8]:
print("Ollama avg latency:", round(res['local_s'].mean(), 2), "s")
print("Gemini avg latency:", round(res['cloud_s'].mean(), 2), "s")
print("Ollama accuracy:", (res['local_label'] == res['true']).mean())
print("Gemini accuracy:", (res['cloud_label'] == res['true']).mean())
print("Agreement between models:", (res['local_label'] == res['cloud_label']).mean())

print("\nReviews where the models disagreed:")
res[res['local_label'] != res['cloud_label']][['review', 'local', 'cloud']]

Ollama avg latency: 0.49 s
Gemini avg latency: 32.17 s
Ollama accuracy: 1.0
Gemini accuracy: 1.0
Agreement between models: 1.0

Reviews where the models disagreed:


,review,local,cloud


## 8. Chain-of-thought vs direct answer
Correct answer: 3 × 12 = 36, and 36 − 5 = **31**.

In [9]:
q = "A store has 3 boxes of 12 apples each. 5 apples are rotten. How many good apples remain?"

for name, fn in [('Ollama', ask_local), ('Gemini', ask_cloud)]:
    print(f"--- {name} DIRECT ---")
    print(fn(q + " Give only the number."))
    print(f"--- {name} STEP BY STEP ---")
    print(fn("Think step by step, then give your final answer.\n\n" + q))
    print()

--- Ollama DIRECT ---
27
--- Ollama STEP BY STEP ---
To find the total number of apples, we multiply the number of boxes by the number of apples per box:

3 boxes * 12 apples/box = 36 apples

Since 5 apples are rotten, to find out how many good apples there are, we subtract the number of rotten apples from the total number of apples:

36 apples - 5 apples = 31 apples

So, there are 31 good apples remaining.

--- Gemini DIRECT ---


31
--- Gemini STEP BY STEP ---


Step 1: Calculate the total number of apples by multiplying the number of boxes by the apples in each box:
3 boxes × 12 apples = 36 apples.

Step 2: Subtract the number of rotten apples from the total to find the number of good apples:
36 - 5 = 31 apples.

**Final Answer:**
There are **31** good apples remaining.



## 9. Gradio chat demo
Uses `gr.Interface`, which works on older Gradio versions (Python 3.9). Stop the cell with the ■ button before moving on.

In [10]:
import gradio as gr

def chat(message):
    return ask_cloud(message)

gr.Interface(fn=chat, inputs="text", outputs="text",
             title="Week 1 Gemini Chat").launch()

/Users/zadeedhaque/Desktop/COS30018/Week1/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IMPORTANT: You are using gradio version 3.36.1, however version 4.44.1 is available, please upgrade.
--------
Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## 10. Comparison table

| | Ollama (llama3.2:1b) | Gemini 3.8 Flash |
|---|---|---|
| Avg latency (s) | 0.49 | 32.17 (range 2.67 to 61.62) |
| Accuracy (10 reviews) | 10/10 | 10/10 |
| Cost | Free, runs on my laptop | Free tier, rate-limited |
| Output quality | Correct, but inconsistent format ("Negative" vs "Negative.") | Correct and consistent |
| Setup effort | Install Ollama + download model | API key only |
| Reliability | Always available offline | 503 errors at peak demand; old model retired (404) |

## 11. Findings

**Setup complexity:** Local was harder to set up. I had to install Ollama, download the model and keep the server running. Cloud only needed an API key, but I hit an invalid key (400), a busy server (503) and a retired model (404).

**Where the models disagreed:** They agreed on all 10 reviews. However, the dataset contains only negative reviews (99 of 99), so this test cannot show whether either model detects positive sentiment. A balanced dataset would be a fairer test.

**Latency:** Ollama was about 65 times faster (0.49 s vs 32.17 s). Gemini's time varied widely because requests travel over the internet, the model "thinks" before answering, and busy servers trigger retries.

**Chain-of-thought:** (write what you saw in section 8: did both models get 31?)

**Reliability trade-off:** The local model always responded but is much smaller. The cloud model is more capable but depends on the internet and Google's servers.

## Problems and fixes
- `jupyter: command not found` → created a virtual environment and installed the packages.
- `400 API key not valid` → generated a new key in AI Studio and loaded it with `getpass`.
- `503 UNAVAILABLE` → switched to `gemini-3.8-flash` and added automatic retries.
- `gradio has no attribute ChatInterface` → older Gradio on Python 3.9, so used `gr.Interface`.